# 🔄 End-to-End CDC + Incremental Load + SCD Project (Medallion Architecture)

> **Goal:** Build a Bronze → Silver → Gold pipeline that ingests **Change Data Capture (CDC)** events, applies **incremental loads**, and maintains an **SCD Type 2** history in the Gold layer — demonstrated with **5 sample records**. Includes interview practice notes at the end.

---

## 📌 Core Concepts (Read First)

| Concept | Meaning |
|---|---|
| **CDC** | Capturing INSERT / UPDATE / DELETE events from a source instead of re-reading the whole table |
| **Incremental Load** | Processing only *new/changed* rows (e.g., new files, `_commit_version`, watermark) instead of a full refresh |
| **SCD Type 1** | Keep only the **latest state** of a record (updates overwrite) |
| **SCD Type 2** | Keep **full history** — every change creates a new versioned row with `__START_AT` / `__END_AT` |
| **Delta Change Data Feed (CDF)** | Delta Lake feature that emits row-level change events (`_change_type`, `_commit_version`, `_commit_timestamp`) |
| **Bronze** | Raw, immutable landing zone (as-is source data) |
| **Silver** | Cleaned, deduplicated, validated data (current state) |
| **Gold** | Business-level aggregates + slowly changing dimension history |

### CDC Change Event Anatomy

Every CDC event has three parts:
1. **Key(s)** — which record changed (e.g., `customer_id`)
2. **Operation** — `insert`, `update`, `delete`
3. **Sequence** — an ordering column so out-of-order events are applied correctly

---

## 🎬 Scenario: Customer Dimension with 5 Sample Records

An operational system emits customer change events. We will:
- **Bronze:** land raw events
- **Silver:** incremental merge to current state (SCD Type 1)
- **Gold:** SCD Type 2 history + aggregate

### The 5 Sample CDC Events (arrive in this order)

| seq | customer_id | name   | city       | status | change_type |
|----:|------------:|--------|------------|--------|-------------|
| 1   | 101         | Alice  | Bangalore  | active | insert      |
| 2   | 102         | Bob    | Delhi      | active | insert      |
| 3   | 3           | Carol  | Mumbai     | active | insert      |
| 4   | 2           | Bob    | Pune       | active | update      |
| 5   | 1           | Alice  | Bangalore  | churned| update      |

**Story:** Bob moves Delhi → Pune (his city changes), and Alice churns. If a 6th event were `delete` for Carol, she would disappear from Silver (SCD1) but her Gold history remains.

---

## 🥉 Step 1 — Bronze Layer (Raw CDC Landing)

**Purpose:** Ingest raw change events exactly as they arrive — no cleaning, no dedup. Bronze is append-only.

```sql
-- Bronze: raw CDC events land here (immutable, append-only)
CREATE TABLE IF NOT EXISTS bronze_customer_cdc (
  seq            BIGINT,      -- event ordering
  customer_id    BIGINT,      -- business key
  name           STRING,
  city           STRING,
  status         STRING,
  change_type    STRING,      -- insert / update / delete
  ingested_at    TIMESTAMP
) USING DELTA;

INSERT INTO bronze_customer_cdc VALUES
  (1, 101, 'Alice', 'Bangalore', 'active',  'insert', current_timestamp()),
  (2, 102, 'Bob',   'Delhi',     'active',  'insert', current_timestamp()),
  (3, 3,   'Carol', 'Mumbai',    'active',  'insert', current_timestamp()),
  (4, 2,   'Bob',   'Pune',      'active',  'update', current_timestamp()),
  (5, 1,   'Alice', 'Bangalore', 'churned', 'update', current_timestamp());
```

**Bronze interview point:** *Why not dedupe in Bronze?* → Bronze must be a faithful, replayable copy of the source. If Silver logic is wrong, you can rebuild Silver/Gold from Bronze without re-hitting the source.

---

## 🥈 Step 2 — Silver Layer (Incremental Clean + SCD Type 1)

**Purpose:** Apply CDC events **incrementally** to maintain the *current state* of each customer.

### Option A — Classic `MERGE` (works everywhere)

```sql
-- Incremental watermark: only process rows newer than the last run
CREATE TABLE IF NOT EXISTS silver_watermark (last_seq BIGINT);
INSERT INTO silver_watermark VALUES (0);

MERGE INTO silver_customer AS t
USING (
  SELECT * FROM bronze_customer_cdc
  WHERE seq > (SELECT last_seq FROM silver_watermark)   -- incremental filter
    AND change_type <> 'delete'                          -- deletes handled below
) s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

-- Handle deletes incrementally
DELETE FROM silver_customer
WHERE customer_id IN (
  SELECT customer_id FROM bronze_customer_cdc
  WHERE seq > (SELECT last_seq FROM silver_watermark)
    AND change_type = 'delete'
);

-- Advance the watermark
UPDATE silver_watermark SET last_seq = (SELECT max(seq) FROM bronze_customer_cdc);
```

### Option B — Modern Auto CDC (single-statement DBSQL/notebook form)

```sql
CREATE OR REFRESH STREAMING TABLE silver_customer
FLOW AUTO CDC
FROM STREAM(bronze_customer_cdc)
KEYS (customer_id)
SEQUENCE BY seq
APPLY AS DELETE WHEN change_type = 'delete'
STORED AS SCD TYPE 1;   -- current state only
```
> ⚠️ Auto CDC streaming sources must be append-only, and in a **notebook/SQL editor you must use this single-statement form** — the two-statement `CREATE FLOW ... INTO` form only works in Lakeflow pipelines. If the source is a Delta CDF stream, exclude the CDF metadata columns: `COLUMNS * EXCEPT (_change_type, _commit_version, _commit_timestamp)`.

### Silver state after the 5 events (SCD Type 1)

| customer_id | name  | city       | status  |
|---:|---|---|---|
| 101 | Alice | Bangalore | churned |
| 102 | Bob   | Pune      | active  |
| 3   | Carol | Mumbai    | active  |

Notice: Bob's "Delhi" row is *gone* — SCD Type 1 keeps only the latest version.

---

## 🥇 Step 3 — Gold Layer (SCD Type 2 History + Aggregates)

### 3a. Dimension with Full History (SCD Type 2 via Auto CDC)

```sql
CREATE OR REFRESH STREAMING TABLE gold_customer_history
FLOW AUTO CDC
FROM STREAM(bronze_customer_cdc)
KEYS (customer_id)
SEQUENCE BY seq
APPLY AS DELETE WHEN change_type = 'delete'
STORED AS SCD TYPE 2;   -- every change = new versioned row
```

Databricks auto-manages `__START_AT` / `__END_AT` validity columns.

### Gold history after the 5 events (SCD Type 2)

| customer_id | name  | city      | status  | __START_AT | __END_AT |
|---:|---|---|---|---:|---:|
| 101 | Alice | Bangalore | active  | 1 | 5 |
| 101 | Alice | Bangalore | churned | 5 | NULL |
| 102 | Bob   | Delhi     | active  | 2 | 4 |
| 102 | Bob   | Pune      | active  | 4 | NULL |
| 3   | Carol | Mumbai    | active  | 3 | NULL |

**As-of query** — "What did we know about Bob at sequence 3?" → `Delhi, active`:

```sql
SELECT * FROM gold_customer_history
WHERE customer_id = 102 AND __START_AT <= 3 AND (__END_AT > 3 OR __END_AT IS NULL);
```

> 💡 If events also carry a *recorded-at* timestamp distinct from the business-time `seq`, you can add `SYSTEM SEQUENCE BY` and `STORED AS BITEMPORAL` to also answer *"what did we believe at time T?"* — two independent time axes.

### 3b. Gold Fact / Aggregate (built incrementally from Silver)

```sql
CREATE OR REFRESH STREAMING TABLE gold_city_summary AS
SELECT city, status, count(*) AS customers
FROM STREAM(silver_customer)
GROUP BY city, status;
```

| city      | status  | customers |
|---|---|---:|
| Bangalore | churned | 1 |
| Pune      | active  | 1 |
| Mumbai    | active  | 1 |

---

## 🏗️ Full Pipeline Diagram

```
Source DB (CDC)
   │  insert/update/delete events (seq, key, op)
   ▼
🥉 BRONZE  bronze_customer_cdc        (raw, append-only, replayable)
   │  incremental: seq > watermark (or STREAM / CDF)
   ▼
🥈 SILVER  silver_customer            (SCD Type 1 — current state, deduped)
   │  full event stream (not deduped)
   ▼
🥇 GOLD    gold_customer_history       (SCD Type 2 — versioned history)
          gold_city_summary             (incremental aggregates)
```

---

## 🧠 Interview Practice — Rapid Fire

1. **What is CDC and why not just full loads?** → Row-level change capture; avoids re-processing entire tables, reduces cost/latency, preserves delete/update semantics that full loads blur.
2. **SCD1 vs SCD2 vs SCD3?** → SCD1: overwrite (latest only). SCD2: full history via new row + validity dates. SCD3: keep only previous value (one extra column).
3. **How do you handle late-arriving events?** → Order by a monotonic `SEQUENCE BY` column; Auto CDC applies events by sequence, not arrival order. For classic MERGE, pick the row with max seq per key before merging.
4. **How do you handle deletes?** → `APPLY AS DELETE WHEN change_type = 'delete'` (Auto CDC), or a separate `DELETE ... WHERE key IN (...)` step (classic).
5. **Incremental vs streaming?** → Incremental = batch job with a watermark (last seq / timestamp / file). Streaming = continuous, serverless-friendly, sub-minute latency.
6. **Why is Bronze immutable?** → Replayability & audit; rebuild Silver/Gold without touching the source.
7. **CDF vs CDC?** → CDC is the *concept* (change capture from source). CDF is Delta Lake's *mechanism* that emits changes (`_change_type`, `_commit_version`) from a Delta table.
8. **How do you detect duplicates in Silver?** → `ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY seq DESC) = 1` — take the latest event per key.
9. **What is the watermark pattern's failure mode?** → If the job crashes after processing but before advancing the watermark, rows process twice — make the advance idempotent (or rely on streaming checkpoints).
10. **When would you choose bitemporal?** → When you need both *business time* (when the fact was true) and *system time* (when we recorded it) — e.g., retroactive corrections, as-of-what-we-knew queries.

### Whiteboard Walkthrough (60-second pitch)
> "I land raw CDC events in Bronze append-only. Silver applies them incrementally — either a MERGE driven by a seq watermark or Auto CDC with KEYS + SEQUENCE BY — to hold the current state, SCD Type 1. Gold splits into two: an SCD Type 2 history table (Auto CDC with `__START_AT`/`__END_AT` so I can query any point in time) and incrementally maintained aggregates. Out-of-order events are safe because everything is sequence-driven."

---

## ✅ Practice Checklist

- [ ] Insert the 5 events into Bronze
- [ ] Verify Silver holds exactly 3 rows (latest per customer)
- [ ] Verify Gold history holds 5 rows (2 versions for Alice, 2 for Bob, 1 for Carol)
- [ ] Run the as-of query for Bob at seq 3 → expect `Delhi / active`
- [ ] Add a 6th event: `(6, 3, 'Carol', 'Mumbai', 'active', 'delete')` → Carol vanishes from Silver, history remains in Gold
- [ ] Re-run the pipeline → watermark means only the new event is processed
